# Reproducing the CPGF paper's results from raw AGFB measurements

This notebook is a companion document to the Circular Polynomial Gradient Filter
with closed-form design laws paper. Every headline number in the paper is a
reduction of the raw per-image measurements stored in `runs/` as Parquet files.
Here we reload those files, recompute each number from scratch with the same
scoring protocol, and place the recomputed value next to the value printed in
the paper. A reader can run this end to end and confirm the figure and table
results.

**How to run.** From the repository root:

```bash
uv sync
uv run jupyter lab reproduce_paper_claims.ipynb   # or: Run All
```

The notebook reads only the `runs/` Parquet files; it writes nothing and needs
no GPU.

**Scoring protocol.** Each filter produces one metric value per image (per
generator cell, seed, and noise condition). Synthetic reductions first collapse
historical duplicate rows for the same cell/filter/metric key, so the clean and
AWGN studies use the 559 unique-cell catalog. A reported number is then the mean
of that value over all cells and seeds for the filter. The single exception is
`noise_gain`, which uses the outlier-robust **median**, matching the benchmark's
own reduction. Wall-clock studies store one timing per configuration directly.

**Reading the verdict tables.** Each section ends in a table with `paper`,
`recomputed`, `abs_diff`, and `match`. Tolerances follow the precision the paper
prints (e.g. an NRMSE quoted to three decimals is checked to 0.001). A `FAIL`
row is a genuine disagreement between the manuscript and the released data, not
a rounding artifact.


In [1]:
import glob
import re

import polars as pl
from IPython.display import display

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_fmt_str_lengths(120)

RUNS = "runs"
CHECKS = []  # (section, quantity, paper, recomputed, passed)


def load(subdir: str) -> pl.DataFrame:
    """Concatenate every Parquet shard for one study under runs/<subdir>/."""
    files = sorted(glob.glob(f"{RUNS}/{subdir}/*.parquet"))
    if not files:
        raise FileNotFoundError(f"no Parquet files under {RUNS}/{subdir}/")
    return pl.concat([pl.read_parquet(f) for f in files], how="diagonal")


def deduplicate_synthetic_results(df: pl.DataFrame) -> pl.DataFrame:
    """Collapse repeated synthetic measurements before computing catalog means."""
    if "cell_id" not in df.columns or "value" not in df.columns:
        return df
    keys = [c for c in df.columns if c not in {"value", "is_nan"}]
    return df.unique(subset=keys, maintain_order=True)


def load_synthetic(subdir: str) -> pl.DataFrame:
    return deduplicate_synthetic_results(load(subdir))


def reduce_metric(df: pl.DataFrame, metric: str, how: str = "mean") -> pl.DataFrame:
    """One value per image -> averaged over all cells/seeds per filter.

    Mirrors the AGFB reporting protocol: NaN-flagged rows are dropped, and
    noise_gain is reduced with the median rather than the mean.
    """
    d = df.filter((pl.col("metric") == metric) & (~pl.col("is_nan")))
    agg = pl.col("value").median() if how == "median" else pl.col("value").mean()
    return d.group_by("filter_config_id", "filter_family").agg(agg.alias(metric))


def radius(cid: str):
    """Support radius parsed from a CPGF config id, e.g. cpgf_radius11_degree1 -> 11."""
    m = re.search(r"radius(\d+)", cid)
    return int(m.group(1)) if m else None


def degree(cid: str):
    m = re.search(r"degree(\d+)", cid)
    return int(m.group(1)) if m else None


def is_even_degree_redundant(cid: str) -> bool:
    m = re.search(r"radius(\d+)_degree(\d+)", cid)
    return (
        m is not None
        and (cid.startswith("cpgf") or cid.startswith("savitzky_golay"))
        and int(m.group(2)) % 2 == 0
    )


def nonredundant_rows(df: pl.DataFrame) -> pl.DataFrame:
    return df.filter(
        ~pl.col("filter_config_id").map_elements(is_even_degree_redundant, return_dtype=pl.Boolean)
    )


def _fmt(x):
    if x is None:
        return "--"
    if isinstance(x, bool):
        return str(x)
    if isinstance(x, float):
        return f"{x:.4g}"
    return str(x)


def compare(section: str, rows: list[tuple]) -> pl.DataFrame:
    """Build a verdict table from (quantity, paper, recomputed, tol) rows.

    A numeric `tol` checks |paper - recomputed| <= tol. Pass tol=None to require
    an exact match (used for identities such as "which filter wins"). Every row
    is also appended to the global CHECKS scorecard.
    """
    table = []
    for quantity, paper, got, tol in rows:
        if tol is None:
            passed = paper == got
            diff = None
        else:
            both = paper is not None and got is not None
            diff = abs(paper - got) if both else None
            passed = bool(both and diff <= tol)
        CHECKS.append((section, quantity, paper, got, passed))
        table.append(
            {
                "quantity": quantity,
                "paper": _fmt(paper),
                "recomputed": _fmt(got),
                "abs_diff": _fmt(diff),
                "match": "PASS" if passed else "FAIL",
            }
        )
    return pl.DataFrame(table)

## 0. What is in the box

Before checking any claim, list the raw result files the rest of the notebook
draws on: how many shards each study has, how many rows, how many random seeds,
and which metrics are present. Nothing here is reduced yet.

In [2]:
studies = {
    "synthetic/clean_accuracy": "clean field, full catalog (Table 2)",
    "synthetic/noise_breadth": "categorical noise bank, 21 models",
    "synthetic/cpgf_grid": "CPGF radius x degree grid under noise",
    "timing/walltime_scaling": "per-filter wall-clock at 4096^2",
    "timing/backend_timing": "forced execution-path sweep",
    "realimg/edges": "BSDS500 / DRIVE / BBBC039 edge ODS",
    "realimg/supersampled": "8x supersampled BSDS500",
}

inventory = []
for sub, desc in studies.items():
    files = sorted(glob.glob(f"{RUNS}/{sub}/*.parquet"))
    df = pl.concat([pl.read_parquet(f) for f in files], how="diagonal")
    inventory.append(
        {
            "study": sub,
            "describes": desc,
            "files": len(files),
            "rows": df.height,
            "seeds": df["seed"].n_unique() if "seed" in df.columns else "-",
            "metrics": (
                ", ".join(sorted(df["metric"].unique().to_list()))
                if "metric" in df.columns
                else "(timing only)"
            ),
        }
    )

pl.DataFrame(inventory)

study,describes,files,rows,seeds,metrics
str,str,i64,i64,str,str
"""synthetic/clean_accuracy""","""clean field, full catalog (Table 2)""",1,625900,"""1""","""angular_mae, edge_fwhm, localization_offset, magnitude_bias, noise_gain, nrmse, sidelobe_ratio, tail_spurious_grad, tail…"
"""synthetic/noise_breadth""","""categorical noise bank, 21 models""",8,3819648,"""8""","""angular_mae, magnitude_bias, noise_gain, nrmse, tail_spurious_grad, tail_vector_error, tangential_normal_leak"""
"""synthetic/cpgf_grid""","""CPGF radius x degree grid under noise""",8,6717312,"""8""","""angular_mae, magnitude_bias, noise_gain, nrmse, tail_spurious_grad, tail_vector_error, tangential_normal_leak"""
"""timing/walltime_scaling""","""per-filter wall-clock at 4096^2""",1,110,"""-""","""(timing only)"""
"""timing/backend_timing""","""forced execution-path sweep""",1,5460,"""-""","""(timing only)"""
"""realimg/edges""","""BSDS500 / DRIVE / BBBC039 edge ODS""",4,4550,"""-""","""ap, auc, dice, ods, ois, orientation_mae"""
"""realimg/supersampled""","""8x supersampled BSDS500""",4,704,"""-""","""ap, ods, ois, orientation_mae, status_only"""


## 1. Clean-field headline accuracy (Table 2, Sec. 6.2)

Every filter is run over the full generator catalog at $4096^2$ with no injected
noise. The paper reports 559 unique generator cells per filter and ranks
representative methods by NRMSE, angular MAE, and magnitude bias. We recompute
all three for the eight filters in the headline table after collapsing duplicate
synthetic rows by cell.


In [3]:
A = load_synthetic("synthetic/clean_accuracy")
n_filters = A["filter_config_id"].n_unique()
cells_per_filter = A.filter(pl.col("metric") == "nrmse").height // n_filters

nrmse = reduce_metric(A, "nrmse")
angmae = reduce_metric(A, "angular_mae")
magbias = reduce_metric(A, "magnitude_bias")
clean = nrmse.join(angmae, on=["filter_config_id", "filter_family"]).join(
    magbias.select("filter_config_id", "magnitude_bias"), on="filter_config_id"
)


def clean_val(cid, col):
    row = clean.filter(pl.col("filter_config_id") == cid)
    return row[col][0] if row.height else None


# Paper Table 2: filter_config_id -> (NRMSE, angular MAE, magnitude bias).
PAPER_T2 = {
    "cpgf_radius3_degree5": (0.112, 21.5, 0.016),
    "central_difference": (0.163, 6.65, -0.008),
    "scharr_3": (0.193, 4.66, -0.012),
    "sobel_3": (0.205, 5.19, -0.012),
    "sobel_7": (0.399, 3.57, -0.036),
    "cpgf_radius3_degree1": (0.448, 3.56, -0.041),
    "derivative_of_gaussian_sigma2": (0.595, 2.82, -0.099),
    "deriche_recursive_gaussian_derivative_sigma4": (0.806, 2.59, -0.220),
}

rows = [("unique cells per filter", 559, cells_per_filter, 0)]
for cid, (p_nrmse, p_angmae, p_magbias) in PAPER_T2.items():
    rows.append((f"{cid}  NRMSE", p_nrmse, clean_val(cid, "nrmse"), 0.001))
    rows.append((f"{cid}  ang. MAE", p_angmae, clean_val(cid, "angular_mae"), 0.05))
    rows.append((f"{cid}  mag. bias", p_magbias, clean_val(cid, "magnitude_bias"), 0.001))

# The paper also claims the catalog-wide NRMSE winner is CPGF r3,d5.
rows.append(
    (
        "best-NRMSE filter (whole catalog)",
        "cpgf_radius3_degree5",
        clean.sort("nrmse").row(0)[0],
        None,
    )
)

compare("1. Clean headline (Table 2)", rows)

quantity,paper,recomputed,abs_diff,match
str,str,str,str,str
"""unique cells per filter""","""559""","""559""","""0""","""PASS"""
"""cpgf_radius3_degree5 NRMSE""","""0.112""","""0.1122""","""0.0001622""","""PASS"""
"""cpgf_radius3_degree5 ang. MAE""","""21.5""","""21.51""","""0.008924""","""PASS"""
"""cpgf_radius3_degree5 mag. bias""","""0.016""","""0.01553""","""0.0004712""","""PASS"""
"""central_difference NRMSE""","""0.163""","""0.1626""","""0.0004291""","""PASS"""
"""central_difference ang. MAE""","""6.65""","""6.65""","""0.000198""","""PASS"""
"""central_difference mag. bias""","""-0.008""","""-0.00792""","""8.04e-05""","""PASS"""
"""scharr_3 NRMSE""","""0.193""","""0.193""","""4.609e-05""","""PASS"""
"""scharr_3 ang. MAE""","""4.66""","""4.656""","""0.003855""","""PASS"""


## 2. Robustness across noise (Sec. 6.3)

Parameters are held fixed and the noise bank is swept over 21 models, 98 strength
conditions, and 8 seeds. The paper states that pooling all conditions, derivative
of Gaussian $\sigma 3$ leads at NRMSE 0.882 with CPGF $r11,d3$ a tie at 0.892.
The CPGF noise-gain collapse with radius (at degree 1) is taken from the
radius$\times$degree grid study, the same source the paper's design-law discussion uses.

In [4]:
C = load_synthetic("synthetic/noise_breadth")
pooled = reduce_metric(C, "nrmse").sort("nrmse")

print("Pooled-noise NRMSE leaderboard (top 5):")
display(pooled.head(5))


def pooled_val(cid):
    row = pooled.filter(pl.col("filter_config_id") == cid)
    return row["nrmse"][0] if row.height else None


# Noise-gain vs radius at degree 1, from the CPGF grid study (median reduction).
CG = load_synthetic("synthetic/cpgf_grid")
ng = reduce_metric(CG.filter(pl.col("filter_family") == "cpgf"), "noise_gain", "median")
ng = ng.with_columns(
    pl.col("filter_config_id").map_elements(radius, return_dtype=pl.Int64).alias("r"),
    pl.col("filter_config_id").map_elements(degree, return_dtype=pl.Int64).alias("d"),
)


def noise_gain(r, d=1):
    row = ng.filter((pl.col("r") == r) & (pl.col("d") == d))
    return row["noise_gain"][0] if row.height else None


rows = [
    ("noise models", 21, C["noise_model"].n_unique(), 0),
    ("strength conditions", 98, C["noise_condition_id"].n_unique(), 0),
    ("seeds", 8, C["seed"].n_unique(), 0),
    ("best-noise filter", "derivative_of_gaussian_sigma3", pooled.row(0)[0], None),
    ("best-noise NRMSE", 0.882, pooled_val("derivative_of_gaussian_sigma3"), 0.001),
    ("CPGF r11,d3 NRMSE (near-tie)", 0.892, pooled_val("cpgf_radius11_degree3"), 0.001),
    ("noise gain, r3  d1", 0.092, noise_gain(3), 0.001),
    ("noise gain, r5  d1", 0.030, noise_gain(5), 0.001),
    ("noise gain, r7  d1", 0.016, noise_gain(7), 0.001),
    ("noise gain, r11 d1", 0.0071, noise_gain(11), 0.0005),
    ("noise gain, r15 d1", 0.0036, noise_gain(15), 0.0005),
]
compare("2. Noise robustness", rows)

Pooled-noise NRMSE leaderboard (top 5):


filter_config_id,filter_family,nrmse
str,str,f64
"""derivative_of_gaussian_sigma3""","""derivative_of_gaussian""",0.881837
"""cpgf_radius11_degree3""","""cpgf""",0.891603
"""derivative_of_gaussian_sigma2""","""derivative_of_gaussian""",0.894019
"""freeman_adelson_g1_sigma2""","""freeman_adelson_g1""",0.894019
"""savitzky_golay_radius7_degree3""","""savitzky_golay""",0.937755


quantity,paper,recomputed,abs_diff,match
str,str,str,str,str
"""noise models""","""21""","""21""","""0""","""PASS"""
"""strength conditions""","""98""","""98""","""0""","""PASS"""
"""seeds""","""8""","""8""","""0""","""PASS"""
"""best-noise filter""","""derivative_of_gaussian_sigma3""","""derivative_of_gaussian_sigma3""","""--""","""PASS"""
"""best-noise NRMSE""","""0.882""","""0.8818""","""0.0001627""","""PASS"""
"""CPGF r11,d3 NRMSE (near-tie)""","""0.892""","""0.8916""","""0.0003971""","""PASS"""
"""noise gain, r3 d1""","""0.092""","""0.09214""","""0.0001436""","""PASS"""
"""noise gain, r5 d1""","""0.03""","""0.03015""","""0.0001476""","""PASS"""
"""noise gain, r7 d1""","""0.016""","""0.01609""","""8.5e-05""","""PASS"""


## 3. Wall-clock parity (Sec. 6.3-6.4)

On $4096^2$ images the spatial CPGF cost grows quadratically with radius while the
FFT path is flat. The paper reports the per-filter timings below and argues a
wide-support CPGF on the FFT path is cheaper than a small spatial stencil. These
timings are A100-specific; a reviewer on other hardware should expect the
*ordering* and *scaling* to hold rather than the absolute milliseconds.

In [5]:
D = load("timing/walltime_scaling")


def ms(cid):
    row = D.filter(pl.col("filter_config_id") == cid)
    return row["ms_per_call"][0] if row.height else None


print("CPGF execution path actually used per radius (degree 1):")
cpgf = (
    D.filter(
        (pl.col("filter_family") == "cpgf") & pl.col("filter_config_id").str.contains("degree1")
    )
    .with_columns(pl.col("filter_config_id").map_elements(radius, return_dtype=pl.Int64).alias("r"))
    .sort("r")
    .select("r", "filter_path", "ms_per_call")
)
display(cpgf)

rows = [
    ("CPGF r3 (spatial)", 17.9, ms("cpgf_radius3_degree1"), 0.2),
    ("CPGF r5 (spatial)", 49.5, ms("cpgf_radius5_degree1"), 0.2),
    ("CPGF r7 (spatial)", 90.9, ms("cpgf_radius7_degree1"), 0.3),
    ("CPGF r11 (FFT)", 3.5, ms("cpgf_radius11_degree1"), 0.1),
    ("CPGF r15 (FFT)", 8.0, ms("cpgf_radius15_degree1"), 0.1),
    ("Sobel 3x3", 13.9, ms("sobel_3"), 0.1),
    ("Sobel 7x7", 18.4, ms("sobel_7"), 0.1),
    ("Derivative of Gaussian sigma2", 29.2, ms("derivative_of_gaussian_sigma2"), 0.1),
    ("Deriche sigma4", 667.0, ms("deriche_recursive_gaussian_derivative_sigma4"), 1.0),
]
compare("3. Wall-clock timing", rows)

CPGF execution path actually used per radius (degree 1):


r,filter_path,ms_per_call
i64,str,f64
3,"""SPARSE_OFFSETS""",17.9074
5,"""SPARSE_OFFSETS""",49.4992
7,"""SPARSE_OFFSETS""",90.87
11,"""FFT""",3.4683
15,"""FFT""",8.0011
21,"""FFT""",3.5025
31,"""FFT""",8.1629
45,"""FFT""",8.2647


quantity,paper,recomputed,abs_diff,match
str,str,str,str,str
"""CPGF r3 (spatial)""","""17.9""","""17.91""","""0.0074""","""PASS"""
"""CPGF r5 (spatial)""","""49.5""","""49.5""","""0.0008""","""PASS"""
"""CPGF r7 (spatial)""","""90.9""","""90.87""","""0.03""","""PASS"""
"""CPGF r11 (FFT)""","""3.5""","""3.468""","""0.0317""","""PASS"""
"""CPGF r15 (FFT)""","""8""","""8.001""","""0.0011""","""PASS"""
"""Sobel 3x3""","""13.9""","""13.94""","""0.039""","""PASS"""
"""Sobel 7x7""","""18.4""","""18.4""","""0.0039""","""PASS"""
"""Derivative of Gaussian sigma2""","""29.2""","""29.24""","""0.0419""","""PASS"""
"""Deriche sigma4""","""667""","""667.4""","""0.3675""","""PASS"""


## 4. Backend crossover (Fig. backend, Sec. 6.4)

A dedicated sweep forces every execution path across the radius range. The paper
claims the two spatial backends climb to roughly 7000 ms at $r63$ while the FFT
path stays in a 3-9 ms band, that the FFT path is degree-independent (8.0 ms
across all seven degrees at $r15$), and that its cost scales with pixel count
(0.3 ms at $1024^2$ to 8.0 ms at $4096^2$ for $r15$).

In [6]:
E = pl.read_parquet(f"{RUNS}/timing/backend_timing/backend_timing_sweep.parquet")
cp = E.filter(
    (pl.col("filter_family") == "cpgf")
    & (pl.col("status") == "ok")
    & pl.col("filter_config_id").str.contains("degree1")
).with_columns(pl.col("filter_config_id").map_elements(radius, return_dtype=pl.Int64).alias("r"))


def path_ms(r, path, size=4096):
    row = cp.filter(
        (pl.col("r") == r) & (pl.col("forced_path") == path) & (pl.col("image_size") == size)
    )
    return row["ms_per_call"][0] if row.height else None


# Crossover radius: smallest radius where FFT beats the fastest spatial path at 4096^2.
at4096 = cp.filter(pl.col("image_size") == 4096)
crossover = None
for r in sorted(at4096["r"].unique().to_list()):
    sub = at4096.filter(pl.col("r") == r)
    fft = sub.filter(pl.col("forced_path") == "FFT")["ms_per_call"]
    spatial = sub.filter(pl.col("forced_path").is_in(["SPATIAL_DENSE", "SPARSE_OFFSETS"]))[
        "ms_per_call"
    ]
    if fft.len() and spatial.len() and fft[0] < spatial.min():
        crossover = r
        break

# Degree independence at r15, 4096^2, FFT path.
deg_fft = E.filter(
    (pl.col("status") == "ok")
    & (pl.col("image_size") == 4096)
    & (pl.col("forced_path") == "FFT")
    & pl.col("filter_config_id").str.contains("radius15_")
)["ms_per_call"]

rows = [
    ("dense spatial at r63 ~ 7000 ms", 7000.0, path_ms(63, "SPATIAL_DENSE"), 1500.0),
    ("sparse spatial at r63 ~ 7000 ms", 7000.0, path_ms(63, "SPARSE_OFFSETS"), 1500.0),
    ("FFT at r63 (inside 3-9 ms band)", 9.0, path_ms(63, "FFT"), 3.0),
    ("crossover radius ~ 4", 4, crossover, 1),
    ("FFT r15 at 1024^2", 0.3, path_ms(15, "FFT", 1024), 0.15),
    ("FFT r15 at 4096^2", 8.0, path_ms(15, "FFT", 4096), 0.2),
    ("FFT r15 degree spread (max-min)", 0.0, float(deg_fft.max() - deg_fft.min()), 0.2),
]
compare("4. Backend crossover", rows)

quantity,paper,recomputed,abs_diff,match
str,str,str,str,str
"""dense spatial at r63 ~ 7000 ms""","""7000""","""6911""","""89.44""","""PASS"""
"""sparse spatial at r63 ~ 7000 ms""","""7000""","""7706""","""706.5""","""PASS"""
"""FFT at r63 (inside 3-9 ms band)""","""9""","""8.911""","""0.0887""","""PASS"""
"""crossover radius ~ 4""","""4""","""5""","""1""","""PASS"""
"""FFT r15 at 1024^2""","""0.3""","""0.3144""","""0.0144""","""PASS"""
"""FFT r15 at 4096^2""","""8""","""8.004""","""0.0041""","""PASS"""
"""FFT r15 degree spread (max-min)""","""0""","""0.1498""","""0.1498""","""PASS"""


## 5. Real-image edge detection (Table, Sec. 7.2)

The real-image study is a fixed-catalog region-boundary evaluation. It scores
182 successful edge configurations and reports 136 nonredundant configurations
after omitting parity-redundant local-polynomial rows. Scoring is raw-mode edge
ODS against the region-boundary ground truth. The paper reports the leading
filter per dataset and notes that the top CPGF catalog row is within 0.005 ODS
of the best everywhere.


In [7]:
R = pl.concat([pl.read_parquet(p) for p in glob.glob(f"{RUNS}/realimg/edges/*.parquet")])


def raw_ods(df, dataset):
    sub = df.filter(
        (pl.col("dataset") == dataset)
        & (pl.col("mode") == "raw")
        & (pl.col("metric") == "ods")
        & (pl.col("status") == "ok")
    )
    return (
        sub.group_by("filter_config_id", "filter_family")
        .agg(pl.col("value").mean().alias("ods"))
        .sort("ods", descending=True)
    )


def metric_for(df, dataset, cid, metric, mode="raw"):
    sub = df.filter(
        (pl.col("dataset") == dataset)
        & (pl.col("mode") == mode)
        & (pl.col("metric") == metric)
        & (pl.col("filter_config_id") == cid)
        & (pl.col("status") == "ok")
    )
    return sub["value"].mean() if sub.height else None


# Paper Table: dataset -> (best filter, ODS, OIS, AP).
PAPER_REAL = {
    "bsds500": ("deriche_recursive_gaussian_derivative_sigma4", 0.681, 0.694, 0.690),
    "drive": ("sobel_5", 0.815, 0.817, 0.510),
    "bbbc039": ("cpgf_radius5_degree7", 0.920, 0.921, 0.887),
}

rows = []
for ds, (cid, p_ods, p_ois, p_ap) in PAPER_REAL.items():
    ranked = raw_ods(R, ds)
    lead = ranked.row(0, named=True)
    rows.append((f"{ds}: best filter", cid, lead["filter_config_id"], None))
    rows.append((f"{ds}: best ODS", p_ods, lead["ods"], 0.001))
    rows.append((f"{ds}: OIS of leader", p_ois, metric_for(R, ds, cid, "ois"), 0.001))
    rows.append((f"{ds}: AP of leader", p_ap, metric_for(R, ds, cid, "ap"), 0.001))
    rows.append((f"{ds}: successful runs", 182, ranked.height, 0))
    rows.append((f"{ds}: nonredundant reported rows", 136, nonredundant_rows(ranked).height, 0))

compare("5. Real-image edges", rows)

quantity,paper,recomputed,abs_diff,match
str,str,str,str,str
"""bsds500: best filter""","""deriche_recursive_gaussian_derivative_sigma4""","""deriche_recursive_gaussian_derivative_sigma4""","""--""","""PASS"""
"""bsds500: best ODS""","""0.681""","""0.6811""","""0.0001164""","""PASS"""
"""bsds500: OIS of leader""","""0.694""","""0.6936""","""0.0004364""","""PASS"""
"""bsds500: AP of leader""","""0.69""","""0.69""","""1.625e-05""","""PASS"""
"""bsds500: successful runs""","""182""","""182""","""0""","""PASS"""
"""bsds500: nonredundant reported rows""","""136""","""136""","""0""","""PASS"""
"""drive: best filter""","""sobel_5""","""sobel_5""","""--""","""PASS"""
"""drive: best ODS""","""0.815""","""0.8149""","""6.97e-05""","""PASS"""
"""drive: OIS of leader""","""0.817""","""0.8169""","""0.0001132""","""PASS"""


## 6. Supersampling and effective support (Fig. ss8, Sec. 7.3)

BSDS500 is rerun through an $8\times$ anti-aliased supersampling pipeline and
rescored against the same ground truth. The paper reports that 33 of 128
nonredundant matched configurations improve (median $\Delta$ODS $=-0.043$),
that the wide CPGF radii recover more than 0.17 ODS, and that the native leaders
fall.


In [8]:
Rs = pl.concat([pl.read_parquet(p) for p in glob.glob(f"{RUNS}/realimg/supersampled/*.parquet")])


def bsds_raw_ods(df):
    return (
        df.filter(
            (pl.col("dataset") == "bsds500")
            & (pl.col("mode") == "raw")
            & (pl.col("metric") == "ods")
            & (pl.col("status") == "ok")
        )
        .group_by("filter_config_id", "filter_family")
        .agg(pl.col("value").mean().alias("ods"))
    )


native = bsds_raw_ods(R).rename({"ods": "native"})
ss8 = bsds_raw_ods(Rs).rename({"ods": "ss8"})
m_all = native.join(ss8, on=["filter_config_id", "filter_family"]).with_columns(
    (pl.col("ss8") - pl.col("native")).alias("delta")
)
m = nonredundant_rows(m_all)


def pair(cid):
    row = m.filter(pl.col("filter_config_id") == cid)
    return (row["native"][0], row["ss8"][0]) if row.height else (None, None)


print("Largest supersampling gains among nonredundant matched rows:")
display(m.sort("delta", descending=True).head(6))

r63_native, r63_ss8 = pair("cpgf_radius63_degree1")
d4_native, d4_ss8 = pair("deriche_recursive_gaussian_derivative_sigma4")

rows = [
    ("successful matched runs", 174, m_all.height, 0),
    ("nonredundant matched configurations", 128, m.height, 0),
    ("nonredundant configurations improved", 33, m.filter(pl.col("delta") > 0).height, 0),
    ("median delta ODS", -0.043, m["delta"].median(), 0.001),
    ("CPGF r63 native ODS", 0.443, r63_native, 0.001),
    ("CPGF r63 supersampled ODS", 0.665, r63_ss8, 0.001),
    ("Deriche sigma4 native ODS", 0.681, d4_native, 0.001),
    ("Deriche sigma4 supersampled ODS", 0.580, d4_ss8, 0.001),
]
compare("6. Supersampling", rows)

Largest supersampling gains among nonredundant matched rows:


filter_config_id,filter_family,native,ss8,delta
str,str,f64,f64,f64
"""cpgf_radius63_degree1""","""cpgf""",0.44335,0.664829,0.221479
"""cpgf_radius45_degree1""","""cpgf""",0.468923,0.676361,0.207439
"""cpgf_radius63_degree3""","""cpgf""",0.478739,0.667561,0.188822
"""derivative_of_gaussian_sigma16""","""derivative_of_gaussian""",0.496136,0.670316,0.17418
"""haar_box_gradient_radius32""","""haar_box_gradient""",0.498959,0.669299,0.17034
"""cpgf_radius31_degree1""","""cpgf""",0.506654,0.665035,0.158381


quantity,paper,recomputed,abs_diff,match
str,str,str,str,str
"""successful matched runs""","""174""","""174""","""0""","""PASS"""
"""nonredundant matched configurations""","""128""","""128""","""0""","""PASS"""
"""nonredundant configurations improved""","""33""","""33""","""0""","""PASS"""
"""median delta ODS""","""-0.043""","""-0.04301""","""1.063e-05""","""PASS"""
"""CPGF r63 native ODS""","""0.443""","""0.4433""","""0.0003499""","""PASS"""
"""CPGF r63 supersampled ODS""","""0.665""","""0.6648""","""0.0001711""","""PASS"""
"""Deriche sigma4 native ODS""","""0.681""","""0.6811""","""0.0001164""","""PASS"""
"""Deriche sigma4 supersampled ODS""","""0.58""","""0.5797""","""0.0003005""","""PASS"""


## 7. Scorecard

The tally of every check above. Any `FAIL` rows are listed so a reviewer sees
exactly where the manuscript and the released data disagree.

In [9]:
score = pl.DataFrame(
    {
        "section": [c[0] for c in CHECKS],
        "quantity": [c[1] for c in CHECKS],
        "paper": [_fmt(c[2]) for c in CHECKS],
        "recomputed": [_fmt(c[3]) for c in CHECKS],
        "match": ["PASS" if c[4] else "FAIL" for c in CHECKS],
    }
)

n_total = score.height
n_pass = score.filter(pl.col("match") == "PASS").height
print(f"{n_pass} / {n_total} checks reproduce the paper to its stated precision.")

fails = score.filter(pl.col("match") == "FAIL")
if fails.height:
    print(f"\n{fails.height} disagreement(s):")
    display(fails)
else:
    print("No disagreements.")

score

79 / 79 checks reproduce the paper to its stated precision.
No disagreements.


section,quantity,paper,recomputed,match
str,str,str,str,str
"""1. Clean headline (Table 2)""","""unique cells per filter""","""559""","""559""","""PASS"""
"""1. Clean headline (Table 2)""","""cpgf_radius3_degree5 NRMSE""","""0.112""","""0.1122""","""PASS"""
"""1. Clean headline (Table 2)""","""cpgf_radius3_degree5 ang. MAE""","""21.5""","""21.51""","""PASS"""
"""1. Clean headline (Table 2)""","""cpgf_radius3_degree5 mag. bias""","""0.016""","""0.01553""","""PASS"""
"""1. Clean headline (Table 2)""","""central_difference NRMSE""","""0.163""","""0.1626""","""PASS"""
"""1. Clean headline (Table 2)""","""central_difference ang. MAE""","""6.65""","""6.65""","""PASS"""
"""1. Clean headline (Table 2)""","""central_difference mag. bias""","""-0.008""","""-0.00792""","""PASS"""
"""1. Clean headline (Table 2)""","""scharr_3 NRMSE""","""0.193""","""0.193""","""PASS"""
"""1. Clean headline (Table 2)""","""scharr_3 ang. MAE""","""4.66""","""4.656""","""PASS"""
